# Ingesta y Auditoría de Microdatos ENAHO (2024 - 2025)
## Módulos 01, 02, 03, 05 y 34 — Lima Metropolitana y Callao (`DOMINIO == 8`)

> **Curso:** Inteligencia Artificial (1INF24) — Pontificia Universidad Católica del Perú (PUCP)  
> **Objetivo de este Notebook:** Cargar, auditar y verificar la integridad de los microdatos de la ENAHO (INEI) para 2024 y 2025, estandarizando las columnas técnicas a **nombres descriptivos según los Diccionarios Metodológicos Oficiales del INEI**, garantizando lectura precisa y cero imputación prematura.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Configurar acceso a src/
ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.io_enaho import (
    load_modulo34,
    load_modulo01,
    load_modulo02,
    load_modulo03,
    load_modulo05,
    KEY_HOGAR,
    KEY_PERSONA,
)

print(f"✓ Entorno configurado correctamente. Raíz del proyecto: {ROOT_DIR}")

---  
## 1. Módulo 34: Sumaria (Resumen de Hogares, Pobreza y Target)

- **Nivel:** Hogar (`conglomerado`, `vivienda`, `hogar`).
- **Filtro:** `dominio == 8` (Lima Metropolitana y Callao).
- **Columnas estandarizadas:** `pobreza_oficial`, `target_pobreza` (1 = Pobre, 0 = No pobre), `factor_expansion`, `miembros_hogar`, `gasto_anual_hogar`, `linea_pobreza`, `lngpc`.

In [ ]:
m34_2024 = load_modulo34(2024, data_root=ROOT_DIR / "Data")
m34_2025 = load_modulo34(2025, data_root=ROOT_DIR / "Data")

print(f"[Módulo 34 - 2024] Hogares: {len(m34_2024):,d} | Pobres: {m34_2024['target_pobreza'].sum():,d} ({m34_2024['target_pobreza'].mean():.2%})")
print(f"[Módulo 34 - 2025] Hogares: {len(m34_2025):,d} | Pobres: {m34_2025['target_pobreza'].sum():,d} ({m34_2025['target_pobreza'].mean():.2%})")
display(m34_2024.head(5))

---  
## 2. Módulo 01: Características de la Vivienda y del Hogar

- **Nivel:** Hogar.
- **Filtro:** `resultado_encuesta in [1, 2]`.
- **Columnas estandarizadas:** `tipo_vivienda`, `material_pared`, `material_piso`, `total_habitaciones`, `total_dormitorios`, `tenencia_vivienda`, `titulo_propiedad`, `registro_sunarp`, `fuente_agua`, `conexion_bano`, `combustible_cocina`, `tiene_internet`, `es_hogar_secundario`.
- **Diagnóstico Crítico (Observación del Jurado):** Conservación estricta de valores nulos (`NaN`) en hogares secundarios (`es_hogar_secundario == 1`). **Cero rellenado automático `ffill/bfill`**.

In [ ]:
m01_2024 = load_modulo01(2024, data_root=ROOT_DIR / "Data")
m01_2025 = load_modulo01(2025, data_root=ROOT_DIR / "Data")

# Delimitar a hogares de Lima Metropolitana
hogares_lima_2024 = m34_2024[KEY_HOGAR].drop_duplicates()
hogares_lima_2025 = m34_2025[KEY_HOGAR].drop_duplicates()

m01_2024 = m01_2024.merge(hogares_lima_2024, on=KEY_HOGAR, how="inner")
m01_2025 = m01_2025.merge(hogares_lima_2025, on=KEY_HOGAR, how="inner")

print(f"[Módulo 01 - 2024] Registros: {len(m01_2024):,d} | Hogares secundarios: {(m01_2024['es_hogar_secundario'] == 1).sum()}")
print(f"                  Nulos sin imputar en tipo_vivienda:     {m01_2024['tipo_vivienda'].isna().sum()}")
print(f"                  Nulos sin imputar en total_habitaciones: {m01_2024['total_habitaciones'].isna().sum()}")

print(f"[Módulo 01 - 2025] Registros: {len(m01_2025):,d} | Hogares secundarios: {(m01_2025['es_hogar_secundario'] == 1).sum()}")
print(f"                  Nulos sin imputar en tipo_vivienda:     {m01_2025['tipo_vivienda'].isna().sum()}")
print(f"                  Nulos sin imputar en total_habitaciones: {m01_2025['total_habitaciones'].isna().sum()}")
display(m01_2024.head(5))

---  
## 3. Módulo 02: Demografía y Características de los Miembros

- **Nivel:** Persona (`conglomerado`, `vivienda`, `hogar`, `cod_persona`).
- **Filtro:** Residentes habituales (`es_miembro_habitual == 1` y no ausente).
- **Columnas estandarizadas:** `parentesco_jefe`, `sexo` (1: Hombre, 2: Mujer), `edad`, `estado_civil`.

In [ ]:
m02_2024 = load_modulo02(2024, data_root=ROOT_DIR / "Data").merge(hogares_lima_2024, on=KEY_HOGAR, how="inner")
m02_2025 = load_modulo02(2025, data_root=ROOT_DIR / "Data").merge(hogares_lima_2025, on=KEY_HOGAR, how="inner")

print(f"[Módulo 02 - 2024] Residentes: {len(m02_2024):,d} en {m02_2024[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
print(f"[Módulo 02 - 2025] Residentes: {len(m02_2025):,d} en {m02_2025[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
display(m02_2024.head(5))

---  
## 4. Módulo 03: Educación y Capital Humano

- **Nivel:** Persona (miembros $\ge 3$ años).
- **Columnas estandarizadas:** `parentesco_jefe`, `lengua_materna`, `nivel_educativo`, `sabe_leer_escribir`, `asiste_centro_educativo`, `tipo_institucion_educ`.

In [ ]:
m03_2024 = load_modulo03(2024, data_root=ROOT_DIR / "Data").merge(hogares_lima_2024, on=KEY_HOGAR, how="inner")
m03_2025 = load_modulo03(2025, data_root=ROOT_DIR / "Data").merge(hogares_lima_2025, on=KEY_HOGAR, how="inner")

print(f"[Módulo 03 - 2024] Personas evaluadas: {len(m03_2024):,d} en {m03_2024[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
print(f"[Módulo 03 - 2025] Personas evaluadas: {len(m03_2025):,d} en {m03_2025[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
display(m03_2024.head(5))

---  
## 5. Módulo 05: Empleo e Informalidad Laboral

- **Nivel:** Persona (PET $\ge 14$ años).
- **Columnas estandarizadas:** `condicion_actividad` (1: Ocupado), `categoria_ocupacional`, `empresa_tiene_ruc`, `horas_semanales_trabajadas`, `afiliacion_pension` (5: Sin pensión).
- **Blindaje metodológico:** Exclusión estricta de ingresos laborales/no laborales (cero *data leakage*).

In [ ]:
m05_2024 = load_modulo05(2024, data_root=ROOT_DIR / "Data").merge(hogares_lima_2024, on=KEY_HOGAR, how="inner")
m05_2025 = load_modulo05(2025, data_root=ROOT_DIR / "Data").merge(hogares_lima_2025, on=KEY_HOGAR, how="inner")

print(f"[Módulo 05 - 2024] Personas en edad de trabajar: {len(m05_2024):,d} en {m05_2024[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
print(f"[Módulo 05 - 2025] Personas en edad de trabajar: {len(m05_2025):,d} en {m05_2025[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
display(m05_2024.head(5))

---  
## 6. Matriz Consolidada de Auditoría de Microdatos (2024 vs. 2025)

Cotejo de integridad dimensional y validación de unicidad de llaves primarias (`PK`).

In [ ]:
def audit_year_modules(year: int, mods: dict) -> pd.DataFrame:
    rows = []
    for name, df in mods.items():
        pk = KEY_HOGAR if name in ["modulo34", "modulo01"] else KEY_PERSONA
        rows.append({
            "Año": year,
            "Módulo": name,
            "Nivel": "Hogar" if name in ["modulo34", "modulo01"] else "Persona",
            "Filas Totales": len(df),
            "Hogares Únicos": df[KEY_HOGAR].drop_duplicates().shape[0],
            "Columnas": df.shape[1],
            "PK Única": not df.duplicated(subset=pk).any(),
        })
    return pd.DataFrame(rows)

mods_2024_all = {
    "modulo34": m34_2024,
    "modulo01": m01_2024,
    "modulo02": m02_2024,
    "modulo03": m03_2024,
    "modulo05": m05_2024,
}

mods_2025_all = {
    "modulo34": m34_2025,
    "modulo01": m01_2025,
    "modulo02": m02_2025,
    "modulo03": m03_2025,
    "modulo05": m05_2025,
}

audit_df = pd.concat([audit_year_modules(2024, mods_2024_all), audit_year_modules(2025, mods_2025_all)], ignore_index=True)
display(audit_df)

---
## 7. Consolidación del Big DataFrame a Nivel Hogar (Operador $\Phi$)

Integración horizontal de los 5 módulos en una sola tabla tabular a nivel hogar utilizando `src.dataset.build_master_dataset()`:
- **Reducción relacional:** Miembros colapsados a hogar (rama de jefatura y funcionales agregados de menores, adultos mayores y educación).
- **Fusión horizontal:** Enlace con Módulo 34 y Módulo 01 por la llave primaria `['conglomerado', 'vivienda', 'hogar']`.
- **Blindaje Anti-Fuga (OOT):** Purga estricta de los 930 hogares panel en 2025 ($N_{\text{test}} = 3{,}199$).
- **Delimitación estricta:** Cero preprocesamiento todavía (nulos, outliers y formatos se mantienen intactos para la fase subsiguiente).

In [ ]:
from src.dataset import build_master_dataset

df_2024 = build_master_dataset(2024, data_root=ROOT_DIR / "Data")
df_2025_test = build_master_dataset(2025, data_root=ROOT_DIR / "Data", purge_panel_2025=True)

print(f"✓ Big DF 2024 (Train/Val): {df_2024.shape} | Pobres: {df_2024['target_pobreza'].sum():,d} ({df_2024['target_pobreza'].mean():.2%})")
print(f"✓ Big DF 2025 (Test OOT):  {df_2025_test.shape} | Pobres: {df_2025_test['target_pobreza'].sum():,d} ({df_2025_test['target_pobreza'].mean():.2%})")
display(df_2024.head(5))

In [ ]:
# Inventario de columnas del Big DataFrame consolidado
print(f"Total variables en el Big DF: {len(df_2024.columns)}")
print(list(df_2024.columns))

---
## 8. Selección Canónica del Espacio de Características ($\mathbf{x}_i \in \mathbb{R}^{28}$)

De las 54 columnas integradas en el Big DataFrame, delimitamos estrictamente las **28 características canónicas** formalizadas en la Sección 3 del informe metodológico:
$$\mathbf{x}_i = [\mathbf{x}_i^{(\text{viv})}, \mathbf{x}_i^{(\text{dem})}, \mathbf{x}_i^{(\text{edu})}, \mathbf{x}_i^{(\text{emp})}] \in \mathbb{R}^{28}$$

### Criterios de Exclusión:
1. **14 variables administrativas y de fuga:** Llaves de identificación (`conglomerado`, `vivienda`, `hogar`), geográficas (`ubigeo`, `dominio`, `estrato`), ponderadores (`factor_expansion`), etiquetas oficiales (`pobreza_oficial`, `target_pobreza`, `y`), variables monetarias de fuga (`gasto_anual_hogar`, `linea_pobreza`, `lngpc`) y temporal (`anio_encuesta`).
2. **5 indicadores NBI agregados:** `nbi1_hacinamiento`, `nbi2_vivienda_inadecuada`, `nbi3_sin_desague`, `nbi4_ninos_sin_escuela`, `nbi5_alta_dependencia_baja_educ` (cuasi-constantes en Lima urbana y redundantes con las variables físicas directas).
3. **7 variables intermedias/redundantes:** `miembros_hogar` (duplicado de `n_miembros`), `jefe_estado_civil` (baja capacidad predictiva de pobreza), etc.

In [ ]:
from src.preprocessing import (
    NUMERIC_COLS,
    CATEGORICAL_COLS,
    CANONICAL_FEATURES_28,
    TARGET_COL,
    GROUP_COL,
    prepare_xy_data,
    SecondaryHouseholdImputer,
    build_preprocessor,
)

print(f"✓ Total características canónicas: {len(CANONICAL_FEATURES_28)}")
print(f"  • Numéricas / Ratios / Ordinales (14): {NUMERIC_COLS}")
print(f"  • Categóricas / Nominales / Binarias (14): {CATEGORICAL_COLS}")

# Verificación de presencia en el dataset de entrenamiento
missing_in_df = [c for c in CANONICAL_FEATURES_28 if c not in df_2024.columns]
assert len(missing_in_df) == 0, f"Variables faltantes: {missing_in_df}"
print("✓ Todas las 28 variables están presentes en df_2024 y df_2025_test.")

---
## 9. Análisis Exploratorio de Datos (EDA) en ENAHO 2024 (Train/Val)

A continuación realizamos el análisis exploratorio formal sobre la muestra de entrenamiento 2024 ($N = 4{,}090$ hogares) para diagnosticar:
1. Balance de clases y factor de desbalance $\gamma$.
2. Auditoría de valores faltantes (distinguiendo la observación OBS-P01 de nulos estructurales).
3. Distribución de valores atípicos y justificación de preservación (sin poda).
4. Correlaciones lineales y monotónicas con el target de pobreza.
5. Frecuencia de categorías raras (< 1%) y justificación del hiperparámetro `min_frequency`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 10, "figure.autolayout": True})

# 9.1. Balance de la variable objetivo
n_total = len(df_2024)
n_poor = int(df_2024[TARGET_COL].sum())
n_non_poor = n_total - n_poor
p_poor = n_poor / n_total
gamma = n_non_poor / n_poor

print("=== DISTRIBUCIÓN DE CLASES (ENAHO 2024) ===")
print(f"Total hogares:       {n_total:,d}")
print(f"Hogares no pobres:   {n_non_poor:,d} ({1 - p_poor:.2%})")
print(f"Hogares pobres (y=1):{n_poor:,d} ({p_poor:.2%})")
print(f"Razón de desbalance: γ ≈ {gamma:.2f} : 1")

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.countplot(x=TARGET_COL, data=df_2024, ax=ax, hue=TARGET_COL, palette=["#2b5c8f", "#d95f02"], legend=False)
ax.set_title(f"Balance de Clases en ENAHO 2024 (Pobreza = {p_poor:.2%}, γ = {gamma:.2f})")
ax.set_xlabel("Target Pobreza Monetaria (0 = No Pobre, 1 = Pobre)")
ax.set_ylabel("Número de Hogares")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,d} ({p.get_height()/n_total:.1%})",
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha="center", va="center", color="white", fontweight="bold")
plt.show()

### 9.2. Auditoría Exhaustiva de Valores Faltantes (Nulos)

Evaluamos los valores faltantes en las 28 variables canónicas antes de cualquier imputación:
- **72 hogares secundarios (1.76%):** Carecen de datos en Módulo 01 (`tipo_vivienda`, `material_pared`, `material_piso`, `total_habitaciones`, `total_dormitorios`). Esto levanta la **OBS-P01**.
- **1,232 jefes inactivos/desempleados:** Tienen nulos en `jefe_horas_semanales`, `jefe_cat_ocupacional` y `jefe_empresa_ruc` porque no trabajan (nulo estructural legítimo).
- **225 hogares sin red de agua continua:** NA en `agua_todos_los_dias`.
- **46 hogares que no cocinan con combustible tradicional:** NA en `combustible_cocina`.

In [ ]:
null_summary = df_2024[CANONICAL_FEATURES_28].isnull().sum()
null_df = pd.DataFrame({
    "Variable": null_summary.index,
    "Nulos": null_summary.values,
    "Porcentaje_Nulos": (null_summary.values / len(df_2024)) * 100
})
null_df = null_df[null_df["Nulos"] > 0].sort_values(by="Nulos", ascending=False).reset_index(drop=True)
display(null_df)

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.barplot(data=null_df, x="Porcentaje_Nulos", y="Variable", ax=ax, hue="Variable", palette="mako", legend=False)
ax.set_title("Porcentaje de Valores Faltantes por Variable (ENAHO 2024)")
ax.set_xlabel("% de Nulos")
ax.set_ylabel("Característica")
plt.show()

### 9.3. Estadísticos Descriptivos y Preservación de Valores Atípicos

De acuerdo con la metodología del proyecto y la recomendación de *Grosh et al. (2022)*, **no se realiza recorte ni poda de valores atípicos** (*outliers*) en tamaño de hogar (`n_miembros`), hacinamiento (`personas_por_hab`) ni tasa de dependencia (`tasa_dependencia`). Podar estos extremos excluiría artificialmente a los hogares más vulnerables y desfavorecidos de Lima Metropolitana. La robustez frente a colas pesadas se garantiza mediante:
1. Invarianza monotónica al orden en clasificadores basados en árboles (CART, Random Forest, LightGBM).
2. Escalamiento robusto basado en mediana y rango intercuartílico (`RobustScaler`) en modelos lineales y PMT-MCO.

In [ ]:
# Estadísticos descriptivos de las variables numéricas
desc = df_2024[NUMERIC_COLS].describe().T[["count", "mean", "std", "min", "50%", "max"]]
desc.columns = ["Conteo", "Media", "Desv_Est", "Min", "Mediana", "Max"]
display(desc.round(2))

# Visualización de distribuciones y outliers clave
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
key_outlier_cols = ["n_miembros", "personas_por_hab", "tasa_dependencia", "jefe_horas_semanales"]
titles = ["Tamaño del Hogar", "Personas por Hab.", "Tasa Dependencia", "Horas Semanales Jefe"]

for ax, col, title in zip(axes, key_outlier_cols, titles):
    sns.boxplot(y=col, x=TARGET_COL, data=df_2024, ax=ax, hue=TARGET_COL, palette=["#2b5c8f", "#d95f02"], legend=False)
    ax.set_title(title)
    ax.set_xlabel("Pobreza (0=No, 1=Sí)")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 9.4. Correlaciones Lineales con la Pobreza Monetaria

Analizamos la matriz de correlación de Pearson entre las características numéricas y el indicador de pobreza $y_i$.
- **Factores de Riesgo / Vulnerabilidad (+):** `personas_por_hab` (+0.34), `n_menores_14` (+0.33), `n_miembros` (+0.33) y `n_ninos_0_5` (+0.29). A mayor carga infantil y hacinamiento, significativamente mayor probabilidad de pobreza.
- **Factores Protectores (-):** `ocupados_por_miembro` (-0.18), `educ_jefe` (-0.17) y `total_habitaciones` (-0.11). La diversificación laboral y el capital humano del jefe actúan como barreras clave ante la vulnerabilidad económica.

In [ ]:
# Matriz de correlación de Pearson
corr_matrix = df_2024[NUMERIC_COLS + [TARGET_COL]].corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax
)
ax.set_title("Matriz de Correlación Lineal (Pearson) - ENAHO 2024")
plt.show()

# Ranking de correlación con target_pobreza
print("=== CORRELACIÓN CON TARGET_POBREZA ===")
print(corr_matrix[TARGET_COL].sort_values(ascending=False).to_string())

### 9.5. Categorías Raras y Tasa de Pobreza Condicional

Para evitar la explosión dimensional y el sobreajuste (*overfitting*) en categorías con escasa representatividad (< 1% de la muestra, es decir, < 40 observaciones), el preprocesador agrupará categorías infrecuentes mediante el hiperparámetro `min_frequency=20` en `OneHotEncoder`.

In [ ]:
rare_summary = []
for c in CATEGORICAL_COLS:
    vc = df_2024[c].value_counts(dropna=False)
    n_rare = (vc < 40).sum()
    pct_rare = vc[vc < 40].sum() / len(df_2024) * 100
    rare_summary.append({
        "Variable": c,
        "Categorías_Totales": len(vc),
        "Categorías_<40_obs": n_rare,
        "%_Muestra_en_Raras": round(pct_rare, 2)
    })

rare_df = pd.DataFrame(rare_summary).sort_values(by="Categorías_<40_obs", ascending=False).reset_index(drop=True)
display(rare_df)

---
## 10. Limpieza de Datos, Imputación Anti-Fuga y Verificación de Nulos Cero

### 10.1. Política de No Eliminación de Hogares (*Zero Drop-Row Policy*)
En encuestas de hogares orientadas a políticas sociales, **descartar filas con nulos () está metodológicamente prohibido**:
1. Los 72 hogares secundarios de 2024 presentan una tasa de pobreza del **27.8%** (frente al 18.9% global). Descartarlos sesgaría el clasificador e invisibilizaría la pobreza en inquilinatos multifamiliares y quintas.
2. Los 225 hogares sin red de agua continua presentan una tasa de pobreza de casi el **40%**.
3. Por ello, se preserva el **100% de las observaciones** ({\text{train}} = 4{,}090$, {\text{test}} = 3{,}199$) aplicando una estrategia de imputación estructurada en dos fases.

### 10.2. Protocolo de Imputación en Dos Fases:
- **Fase 1 ():** Imputa habitaciones y dormitorios mediante la **mediana del conglomerado censal** ajustada estrictamente sobre el conjunto de entrenamiento (OBS-P01), recalcula el hacinamiento () de manera relacional, y fija en 0.0 las horas de jefes inactivos.
- **Fase 2 ():** Imputa con la mediana de entrenamiento los nulos restantes en numéricas (, ), y asigna el código sentinela  (*"desconocido"*) a las categóricas antes de aplicar la codificación One-Hot con poda de raras (< 1%).

In [ ]:
from src.preprocessing import (
    build_preprocessor,
    SecondaryHouseholdImputer,
    prepare_xy_data,
    CANONICAL_FEATURES_28,
)

# 1. Extracción de variables canónicas (X, y, groups)
X_tr, y_tr, grp_tr = prepare_xy_data(df_2024)
X_te, y_te, grp_te = prepare_xy_data(df_2025_test)

# Conteo inicial de nulos en las 28 variables
nulls_tr_before = X_tr[CANONICAL_FEATURES_28].isnull().sum()
nulls_te_before = X_te[CANONICAL_FEATURES_28].isnull().sum()

# 2. Fase 1: Limpiador de hogares secundarios y estructura física (OBS-P01)
cleaner = SecondaryHouseholdImputer(group_col="conglomerado")
X_tr_clean = cleaner.fit_transform(X_tr)
X_te_clean = cleaner.transform(X_te)

nulls_tr_f1 = X_tr_clean[CANONICAL_FEATURES_28].isnull().sum()
nulls_te_f1 = X_te_clean[CANONICAL_FEATURES_28].isnull().sum()

# 3. Fase 2: ColumnTransformer (Imputación final, escalado y One-Hot Encoding)
preprocessor = build_preprocessor(scale_numeric=True, one_hot=True, min_category_frequency=0.01)
X_tr_proc = preprocessor.fit_transform(X_tr_clean)
X_te_proc = preprocessor.transform(X_te_clean)

# Tabla de auditoría del Data Cleaning: Antes vs. Después
vars_with_nulls = nulls_tr_before[nulls_tr_before > 0].index
cleaning_audit = pd.DataFrame({
    "Variable": vars_with_nulls,
    "Nulos_Train_Antes": nulls_tr_before[vars_with_nulls].values,
    "Nulos_Train_Fase1": nulls_tr_f1[vars_with_nulls].values,
    "Nulos_Test_Antes": nulls_te_before[vars_with_nulls].values,
    "Nulos_Test_Fase1": nulls_te_f1[vars_with_nulls].values,
    "Estrategia_Imputación": [
        "Mediana Conglomerado Censal (OBS-P01)",
        "Mediana Conglomerado Censal (OBS-P01)",
        "Recálculo Relacional: miembros / hab",
        "Mediana Train (54 años)",
        "Mediana Train (6 = Sec. completa)",
        "Mediana Train / 0.0 jefes inactivos",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('sin red pública continua')",
        "Sentinela -1.0 ('no cocina / desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')"
    ]
})

print("=== TABLA DE AUDITORÍA DEL DATA CLEANING (ANTES VS. DESPUÉS) ===")
display(cleaning_audit)

print("=== BALANCE FINAL DE INTEGRIDAD DE DATOS ===")
print(f"✓ Filas preservadas en Train 2024: {len(X_tr_proc):,d} (0 eliminadas)")
print(f"✓ Filas preservadas en Test 2025:  {len(X_te_proc):,d} (0 eliminadas)")
print(f"✓ Nulos finales en Train 2024:      {np.isnan(X_tr_proc).sum()} ({np.isnan(X_tr_proc).sum() == 0})")
print(f"✓ Nulos finales en Test 2025:       {np.isnan(X_te_proc).sum()} ({np.isnan(X_te_proc).sum() == 0})")
print(f"✓ Dimensión final de la matriz:     {X_tr_proc.shape[1]} columnas densas tras One-Hot Encoding")
